# Data Cleaning and Merging Step
Because we have numerous datasets and from different sources, we need to find a way to standardize the data to be able to join them. At a glance, the easiest way to do so is to merge via county name. However, we will see later why this is not the case. We start by first importing the relevant packages and reading in all the CSVs. We also fix the FIPS codes in PopulationEstimates.csv by adding leading zeroes to keep consistency with the other FIPS codes used later.

In [1]:
import pandas as pd
import numpy as np
from functools import reduce

# Load data
pop = pd.read_csv("PopulationEstimates.csv", encoding='latin-1', dtype={'fipstxt': str}).rename(columns=lambda x: x.strip().lower().replace(' ', '_'))
poverty = pd.read_csv("Poverty2023.csv", encoding='latin-1').rename(columns=lambda x: x.strip().lower().replace(' ', '_'))
edu = pd.read_csv("Education2023.csv", encoding='latin-1').rename(columns=lambda x: x.strip().lower().replace(' ', '_'))
unemp = pd.read_csv("Unemployment2023.csv", encoding='latin-1').rename(columns=lambda x: x.strip().lower().replace(' ', '_'))
crime_rate = pd.read_csv('crime_rate.csv', encoding='latin-1')
temperature = pd.read_csv('temperature.csv', encoding='latin-1', comment='#')
precipitation = pd.read_csv('precipitation.csv', encoding='latin-1', comment='#')
type(pop['fipstxt'])
pop['fipstxt'] = pop['fipstxt'].apply(lambda x: f"{int(x):05d}" if pd.notnull(x) else np.nan)
pop['fipstxt'] = pop['fipstxt'].str.zfill(5)


We then subset the datasets by selecting just the attribute that we want, storing it in a new variable, and then standardizing column names by renaming them.

In [2]:

# First subset by the attribute we want, then select the state and area name and create a new variable to store values in
pop = pop[pop['attribute'] == 'POP_ESTIMATE_2023'][['state', 'area_name', 'value','fipstxt']].rename(columns={'value': 'Population_Estimate','fipstxt': 'FP'})
poverty = poverty[poverty['attribute'] == 'PCTPOVALL_2023'][['stabr', 'area_name', 'value']].rename(columns={'stabr': 'state', 'value': 'Poverty_Percent'})
edu = edu[edu['attribute'] == "Bachelor's degree or higher, 2019-23"][['state', 'area_name', 'value']].rename(columns={'value': 'Bachelor_Or_Higher'})

precipitation['state'] = precipitation['ID'].str[:2]
precipitation_cleaned = precipitation[['state', 'Value', 'Name']].rename(columns={'Value': 'Avg_Precipitation', 'Name': 'area_name'})

temperature['state'] = temperature['ID'].str[:2]
temperature_cleaned = temperature[['state', 'Value', 'Name']].rename(columns={'Value': 'Avg_Temp', 'Name': 'area_name'})

crime_cleaned = crime_rate.copy()
crime_cleaned[['area_name', 'state']] = crime_cleaned['county_name'].str.split(", ", expand=True)
crime_cleaned['area_name'] = crime_cleaned['area_name'].str.upper()
crime_cleaned['state'] = crime_cleaned['state'].str.upper()
crime_cleaned['Crime_Rate_Per_100000'] = crime_cleaned['crime_rate_per_100000'].round(2)
crime_cleaned = crime_cleaned[['area_name', 'state', 'Crime_Rate_Per_100000']]


We then do some cleaning of specific datasets to keep them standard with the rest.

In [3]:

# Removing the , XX at the end of area names in unemp
unemp['area_name'] = unemp['area_name'].str.replace(",\\s*\\w{2}$", "", regex=True)
medianincome = unemp[unemp['attribute'] == 'Median_Household_Income_2022'][['state', 'area_name', 'value']].rename(columns={'value': 'Median_Income'})
unemp = unemp[unemp['attribute'] == 'Unemployment_rate_2023'][['state', 'area_name', 'value']].rename(columns={'value': 'Unemployment_Rate'})


Next we do a join of the datasets, merging by area_name and state, which are county names and the two letter abbreviation for a state respectively. Once we're done with the join, we remove any missing values that are present. Since we used multiple sources, some counties got dropped due to some sources having missing values for said county.

In [4]:

# Combines datasets into a list to be used via full join
data_list = [pop, poverty, edu, unemp, medianincome, temperature_cleaned, precipitation_cleaned]

# Combines the data, normalizes the capitalization, and collapses any duplicate entries after basic data cleaning, and finally removes rows with missing values
combined_data = reduce(lambda left, right: pd.merge(left, right, on=['area_name', 'state'], how='outer'), data_list)
combined_data['area_name'] = combined_data['area_name'].str.upper()
combined_data['state'] = combined_data['state'].str.upper()
combined_data = combined_data.merge(crime_cleaned, on=['area_name', 'state'], how='outer')

combined_data['area_name'] = combined_data['area_name'].str.replace(r"/.*", "", regex=True).str.strip()
combined_data = combined_data.groupby(['area_name', 'state']).agg(lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan).reset_index()
combined_data = combined_data.dropna()

print(combined_data.head())

          area_name state  Population_Estimate     FP  Poverty_Percent  \
0  ABBEVILLE COUNTY    SC              24434.0  45001             15.3   
1     ACADIA PARISH    LA              56489.0  22001             25.0   
2   ACCOMACK COUNTY    VA              33239.0  51001             17.3   
3        ADA COUNTY    ID             524673.0  16001              8.0   
4      ADAIR COUNTY    IA               7389.0  19001              9.8   

   Bachelor_Or_Higher  Unemployment_Rate  Median_Income  Avg_Temp  \
0              3332.0                3.7        50325.0      64.4   
1              5071.0                3.7        42981.0      70.3   
2              5314.0                3.5        56357.0      59.1   
3            153355.0                2.7        87748.0      52.8   
4              1052.0                2.4        66997.0      50.6   

   Avg_Precipitation  Crime_Rate_Per_100000  
0              37.73                 511.86  
1              62.29                 163.98  
2 

However, as we add the walkability dataset, this method of joining via county names no longer works since walkability doesn’t have county names in the data. But to even merge the walkability dataset, we need to first convert it from GBD to a CSV so we can actually work with it via dataframes. In the process, we also create the 5 digit FIPS code using values given by the dataset.

In [6]:
import geopandas as gpd
# ADDING WALKABILITY + LAND AREA DATASETS
gdf = gpd.read_file("Natl_WI.gdb", layer="NationalWalkabilityIndex")  # Replace with actual layer name
full_data = gdf.drop(columns=["geometry"])  # drop geometry if you just want tabular data
# We will use STATEFP and COUNTYFP as a "key" to merge with our combined dataset.
# There are 220,000 observations for 3,000 counties.
# We need to average the walkability index (NatWalkInd) among all the blocks in each county.
# Weighted by area (Shape_Area) of the block.
df = full_data[["STATEFP","COUNTYFP","NatWalkInd","Shape_Area"]].copy()
# Combine STATEFP and COUNTYFP to FP.
df["FP"] = df["STATEFP"].astype(str).str.zfill(2) + df["COUNTYFP"].astype(str).str.zfill(3)
df = df.drop(columns=["STATEFP","COUNTYFP"])
import numpy as np
# Weighted average of NatWalkInd by FP
walkability_df = (
    df.groupby("FP")[["NatWalkInd", "Shape_Area"]]
      .apply(lambda g: (g["NatWalkInd"] * g["Shape_Area"]).sum() / g["Shape_Area"].sum())
      .reset_index(name="Walkability")
)


Finally, we take our land area dataset and use it with the population estimate to get a population density, which we will use over population estimate as it fits closer with a user's desire, high population doesn't intrinsically mean high density, the population can be spread out.

In [7]:
area = pd.read_csv("Land_Area.csv", encoding='utf-8-sig')
area.rename(columns={"STCOU":"FP", "LND010190D":"Land_Area"}, inplace=True)
area["FP"] = area["FP"].astype(str).str.zfill(5)

Now that we can verify that our walkability dataset has both the FIPS codes and walkability values, we can finally merge the walkability dataset onto our main dataset using the FIPS code. We save our this version of the combined dataset for sharing.

In [ ]:
merged_df = pd.merge(combined_data, walkability_df, on='FP', how='inner')
merged_df2 = pd.merge(merged_df, area, on='FP', how='inner')
# There was actually one row with a land area of 0. I have edited in the Census value into the CSV beforehand
print(merged_df2.iloc[[277]])
# Adding in population density as a potentially useful metric.
merged_df2["Population_Density"] = merged_df2["Population_Estimate"] / merged_df2["Land_Area"]
merged_df2["Population_Density"]
# Putting it all into a csv.
# merged_df2.to_csv("county.csv", encoding='utf-8-sig') 
print(merged_df2.head())

             area_name state  Population_Estimate     FP  Poverty_Percent  \
277  BROOMFIELD COUNTY    CO              76860.0  08014              5.1   

     Bachelor_Or_Higher  Unemployment_Rate  Median_Income  Avg_Temp  \
277             31270.0                3.0       114746.0      52.7   

     Avg_Precipitation  Crime_Rate_Per_100000  Walkability  Land_Area  
277              10.21                  50.44     8.622119      33.58  
          area_name state  Population_Estimate     FP  Poverty_Percent  \
0  ABBEVILLE COUNTY    SC              24434.0  45001             15.3   
1     ACADIA PARISH    LA              56489.0  22001             25.0   
2   ACCOMACK COUNTY    VA              33239.0  51001             17.3   
3        ADA COUNTY    ID             524673.0  16001              8.0   
4      ADAIR COUNTY    IA               7389.0  19001              9.8   

   Bachelor_Or_Higher  Unemployment_Rate  Median_Income  Avg_Temp  \
0              3332.0                3.7    